# Direction 2 · Phase B · Slot 1: Pre-GAP (`DAR`, `Oracle-Mask-Pool`) & Internal Adaptation Control Ladder

**Settings:** Enable **GPU T4 x1** and **Internet ON**.

**Zero manual uploads required:**
- Clones `spurious_correlation_mechanisms` from GitHub.
- Auto-downloads the 7 saved ResNet-50 `ckpt_last.pt` backbones (`5 standard_erm` + `2 erm_star`) from your public Kaggle outputs.
- Auto-downloads Waterbirds images (~490 MB) + Caltech `CUB_200_2011` pixel masks (~39 MB).
- Runs 5-fold cleaned **validation-only** CV (`split == 1`, zero test access) across `DFR-L2`, `DAR`, `Oracle-Mask-Pool`, `H2T-DFR`, `BN-Stat-Only`, `Affine-DFR`, and `Layer4-FT`.
- Outputs `/kaggle/working/rep_repair_phase_b_controls.zip`.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch

REPO_URL = 'https://github.com/chautanphat/waterbirds-shortcut-mitigation.git'
BRANCH = 'spurious_correlation_mechanisms'
CODE_DIR = Path('/kaggle/working/repo')
WORK_DIR = Path('/kaggle/working/phase_b_assets')
OUT_DIR = Path('/kaggle/working/rep_repair_phase_b_controls')

print('CUDA available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CODE_DIR)], check=True)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Pre-flight unit tests.
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_rep_repair_phase_b.py', '-v'], check=True)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cmd = [
    sys.executable, '-B', '-m', 'scripts.analysis.rep_repair_phase_b_controls',
    '--work-dir', str(WORK_DIR),
    '--output', str(OUT_DIR),
    '--device', device,
    '--workers', '2',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
from IPython.display import Image, Markdown, display

display(Markdown((OUT_DIR / 'README.md').read_text(encoding='utf-8')))
display(Image(filename=str(OUT_DIR / 'phase_b_controls_ladder.png')))
print(json.dumps(json.loads((OUT_DIR / 'gate2_controls_summary.json').read_text()), indent=2))

zip_path = Path('/kaggle/working/rep_repair_phase_b_controls.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.rglob('*')):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(OUT_DIR))
print(f'Saved results bundle: {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)')